<a href="https://colab.research.google.com/github/ManLikePamo/Project/blob/main/App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok pillow tensorflow numpy opencv-python-headless -q
!pip install streamlit pillow tensorflow numpy opencv-python-headless mediapipe -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 148.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.3 MB/s eta 0:00:00


In [2]:
app_code = '''
import streamlit as st
import numpy as np
from PIL import Image
import io
import os

st.set_page_config(page_title="Facial Emotion Recognition", layout="wide")

st.markdown("""
<style>
    .main { background-color: #f0f2f6; }
    .subtitle { text-align: center; color: #666; font-size: 14px;
                margin-top: -10px; margin-bottom: 20px; }
    .step-card {
        background: white; border-radius: 12px; padding: 20px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); text-align: center;
    }
    .step-badge {
        background: #4A90D9; color: white; border-radius: 50%;
        width: 28px; height: 28px; display: inline-flex;
        align-items: center; justify-content: center;
        font-weight: bold; font-size: 14px; margin-bottom: 8px;
    }
    .step-title   { font-weight: 700; font-size: 15px; margin-bottom: 4px; }
    .step-subtitle { color: #888; font-size: 12px; margin-bottom: 12px; }
    .arrow { font-size: 28px; color: #4A90D9; display: flex;
             align-items: center; justify-content: center;
             height: 100%; padding-top: 60px; }
    .success-bar {
        background: #e8f5e9; border: 1px solid #a5d6a7;
        border-radius: 8px; padding: 10px 16px; color: #2e7d32;
        font-size: 14px; text-align: center; margin-top: 16px;
    }
</style>
""", unsafe_allow_html=True)

# ── Load emotion model ─────────────────────────────────────────────
@st.cache_resource
def load_model():
    try:
        import tensorflow as tf
        import os

        # ── Define focal loss to match training ───────────
        def focal_loss(gamma=2.0, alpha=0.25):
            def loss_fn(y_true, y_pred):
                y_true  = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
                y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0)
                probs   = tf.reduce_sum(
                            y_pred * tf.one_hot(y_true, tf.shape(y_pred)[-1]),
                            axis=-1)
                focal_w = tf.pow(1.0 - probs, gamma)
                alpha_w = alpha * tf.cast(y_true > 0, tf.float32) + \
                          (1 - alpha) * tf.cast(y_true == 0, tf.float32)
                return tf.reduce_mean(alpha_w * focal_w * -tf.math.log(probs))
            return loss_fn

        class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
            def __init__(self, gamma=2.0, **kwargs):
                super().__init__(**kwargs)
                self.gamma = gamma
            def get_config(self):
                return {**super().get_config(), "gamma": self.gamma}
            def call(self, y_true, y_pred):
                y_true  = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
                y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0)
                probs   = tf.reduce_sum(
                            y_pred * tf.one_hot(y_true, tf.shape(y_pred)[-1]),
                            axis=-1)
                focal_w = tf.pow(1.0 - probs, self.gamma)
                return tf.reduce_mean(focal_w * -tf.math.log(probs))

        path = "/content/drive/MyDrive/Project/Dataset/model/model_improved.keras"

        model = tf.keras.models.load_model(
            path,
            custom_objects={
                "loss_fn":                    focal_loss(gamma=2.0, alpha=0.25),
                "focal_loss":                 focal_loss,
                "SparseCategoricalFocalLoss": SparseCategoricalFocalLoss,
            }
        )
        return model

    except Exception as e:
        st.warning(f"Model load error: {e}")
        return None


@st.cache_resource
def load_face_detector():
    import cv2
    import urllib.request
    import os

    # ── Download OpenCV DNN face detector ─────────────────
    proto_path = "/content/deploy.prototxt"
    model_path = "/content/res10_300x300_ssd_iter_140000.caffemodel"

    if not os.path.exists(proto_path):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/opencv/opencv/master/"
            "samples/dnn/face_detector/deploy.prototxt",
            proto_path
        )
    if not os.path.exists(model_path):
        urllib.request.urlretrieve(
            "https://github.com/opencv/opencv_3rdparty/raw/"
            "dnn_samples_face_detector_20170830/"
            "res10_300x300_ssd_iter_140000.caffemodel",
            model_path
        )

    net = cv2.dnn.readNetFromCaffe(proto_path, model_path)
    return net

model      = load_model()
face_net   = load_face_detector()

EMOTIONS = ["Angry","Disgust","Fear","Happy","Sad","Surprise","Neutral"]
EMOJIS   = {"Angry":"😠","Disgust":"🤢","Fear":"😨","Happy":"😊",
             "Sad":"😢","Surprise":"😲","Neutral":"😐"}
COLORS   = {"Angry":"#e53935","Disgust":"#43a047","Fear":"#7b1fa2",
             "Happy":"#e53935","Sad":"#1e88e5","Surprise":"#fb8c00","Neutral":"#546e7a"}

# ── Face detection with MediaPipe + Haar fallback ──────────────────
def detect_face(img_rgb: np.ndarray):
    """
    Try MediaPipe first (handles angles, lighting, partial faces).
    Fall back to Haar Cascade if MediaPipe finds nothing.
    Returns: (x, y, w, h) of the best face, or None.
    """
    import cv2

    h_img, w_img = img_rgb.shape[:2]

    # ── MediaPipe detection ────────────────────────────────────────
    results = face_detect.process(img_rgb)

    if results.detections:
        # Pick detection with highest confidence
        best = max(results.detections,
                   key=lambda d: d.score[0])
        bbox = best.location_data.relative_bounding_box
        x = int(bbox.xmin  * w_img)
        y = int(bbox.ymin  * h_img)
        w = int(bbox.width * w_img)
        h = int(bbox.height * h_img)
        # Clamp to image bounds
        x = max(0, x); y = max(0, y)
        w = min(w, w_img - x); h = min(h, h_img - y)
        return x, y, w, h, "mediapipe"

    # ── Haar Cascade fallback ──────────────────────────────────────
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # Try frontal face
    frontal = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    faces = frontal.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3, minSize=(20, 20)
    )
    if len(faces) > 0:
        x, y, w, h = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)[0]
        return x, y, w, h, "haar_frontal"

    # Try profile face (catches side angles)
    profile = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_profileface.xml"
    )
    faces = profile.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3, minSize=(20, 20)
    )
    if len(faces) > 0:
        x, y, w, h = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)[0]
        return x, y, w, h, "haar_profile"

    return None

# ── Full preprocessing pipeline ────────────────────────────────────
def preprocess(image: Image.Image):
    import cv2
    import numpy as np

    img_rgb   = np.array(image.convert("RGB"))
    img_gray  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    h, w      = img_gray.shape

    # ── Detect face with OpenCV DNN ───────────────────────
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    blob    = cv2.dnn.blobFromImage(
        img_bgr, 1.0, (300, 300), (104.0, 177.0, 123.0)
    )
    face_net.setInput(blob)
    detections = face_net.forward()

    face_rect  = None
    best_conf  = 0.0

    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        if conf > 0.4 and conf > best_conf:
            best_conf = conf
            x1 = int(detections[0, 0, i, 3] * w)
            y1 = int(detections[0, 0, i, 4] * h)
            x2 = int(detections[0, 0, i, 5] * w)
            y2 = int(detections[0, 0, i, 6] * h)
            x1 = max(0, x1); y1 = max(0, y1)
            x2 = min(w, x2); y2 = min(h, y2)
            if x2 > x1 and y2 > y1:
                face_rect = (x1, y1, x2, y2)

    if face_rect is not None:
        x1, y1, x2, y2 = face_rect
        fw = x2 - x1; fh = y2 - y1

        # Add padding so forehead and chin are included
        pad_x = int(0.20 * fw); pad_y = int(0.30 * fh)
        x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
        x2 = min(w, x2 + pad_x); y2 = min(h, y2 + pad_y)

        cropped  = img_gray[y1:y2, x1:x2]
        face_pil = Image.fromarray(cropped)
        st.info(f"✅ Face detected and cropped ({fw}×{fh}px)")
    else:
        # Fallback — centre square crop
        size = min(h, w)
        y1   = (h - size) // 2; x1 = (w - size) // 2
        cropped  = img_gray[y1:y1+size, x1:x1+size]
        face_pil = Image.fromarray(cropped)
        st.warning("⚠️ No face detected — using centre crop.")

    resized = face_pil.resize((48, 48), Image.LANCZOS)
    arr     = np.array(resized, dtype=np.float32) / 255.0
    return resized, arr.reshape(1, 48, 48, 1)

def predict(arr):
    if model is not None:
        preds = model.predict(arr, verbose=0)[0]
    else:
        preds = np.random.dirichlet(np.ones(7) * 0.5)
    idx = int(np.argmax(preds))
    return EMOTIONS[idx], float(preds[idx]) * 100, preds

# ── Title ──────────────────────────────────────────────────────────
st.markdown("<h2 style=\\'text-align:center\\'>Facial Emotion Recognition</h2>",
            unsafe_allow_html=True)
st.markdown("<div class=\\'subtitle\\'>FER2013 CNN Model • 7 Emotion Classes</div>",
            unsafe_allow_html=True)

# ── Session state ──────────────────────────────────────────────────
for key in ["cap_img","proc_img","prediction"]:
    if key not in st.session_state:
        st.session_state[key] = None

# ── 4-step layout ──────────────────────────────────────────────────
c1, a1, c2, a2, c3, a3, c4 = st.columns([3, 0.5, 3, 0.5, 3, 0.5, 3])

# Card 1 – capture
with c1:
    st.markdown("""<div class="step-card">
        <div><span class="step-badge">1</span></div>
        <div class="step-title">Webcam / Upload</div>
        <div class="step-subtitle">Capture or upload a face image</div>
    </div>""", unsafe_allow_html=True)
    webcam = st.camera_input("", label_visibility="collapsed")
    upload = st.file_uploader("Or upload image", type=["jpg","jpeg","png"],
                               label_visibility="collapsed")
    src = webcam or upload
    if src:
        img = Image.open(src).convert("RGB")
        st.session_state.cap_img    = img
        g, arr                      = preprocess(img)
        st.session_state.proc_img   = g
        st.session_state.prediction = predict(arr)

# Arrow 1
with a1:
    st.markdown("<div class=\\'arrow\\'>➜</div>", unsafe_allow_html=True)

# Card 2 – captured
with c2:
    st.markdown("""<div class="step-card">
        <div><span class="step-badge">2</span></div>
        <div class="step-title">Captured Image</div>
        <div class="step-subtitle">Image received from webcam</div>
    </div>""", unsafe_allow_html=True)
    if st.session_state.cap_img:
        st.image(st.session_state.cap_img, use_container_width=True)
    else:
        st.markdown("<br><div style=\\'text-align:center;color:#bbb;padding:40px 0\\'>"
                    "📷 Awaiting capture…</div>", unsafe_allow_html=True)

# Arrow 2
with a2:
    st.markdown("<div class=\\'arrow\\'>➜</div>", unsafe_allow_html=True)

# Card 3 – preprocessing
with c3:
    st.markdown("""<div class="step-card">
        <div><span class="step-badge">3</span></div>
        <div class="step-title">Preprocessing</div>
        <div class="step-subtitle">Preparing model input</div>
        <ul style="text-align:left;font-size:13px;color:#444;margin-top:8px;">
            <li>MediaPipe Face Detection</li>
            <li>Convert to Grayscale</li>
            <li>Resize to 48 × 48 pixels</li>
        </ul>
    </div>""", unsafe_allow_html=True)
    if st.session_state.proc_img:
        buf = io.BytesIO()
        st.session_state.proc_img.save(buf, format="PNG")
        col_l, col_c, col_r = st.columns([1, 2, 1])
        with col_c:
            st.image(buf.getvalue(), caption="48×48 – Model Input Ready", width=120)
    else:
        st.markdown("<br><div style=\\'text-align:center;color:#bbb;padding:40px 0\\'>"
                    "🖼 Preview here</div>", unsafe_allow_html=True)

# Arrow 3
with a3:
    st.markdown("<div class=\\'arrow\\'>➜</div>", unsafe_allow_html=True)

# Card 4 – prediction
with c4:
    st.markdown("""<div class="step-card">
        <div><span class="step-badge">4</span></div>
        <div class="step-title">Emotion Prediction</div>
        <div class="step-subtitle">CNN Classification Result</div>
    """, unsafe_allow_html=True)
    if st.session_state.prediction:
        emo, conf, preds = st.session_state.prediction
        col  = COLORS[emo]
        emoj = EMOJIS[emo]
        st.markdown(f"""
        <div style="text-align:center;margin-top:6px;">
            <div style="font-size:12px;color:#777;margin-bottom:4px;">Predicted Emotion</div>
            <div style="background:{col};color:white;border-radius:10px;
                        padding:10px 18px;font-size:20px;font-weight:bold;
                        display:inline-block;">{emoj} {emo}</div>
            <div style="color:#555;font-size:14px;margin-top:6px;">
                Confidence: {conf:.1f}%</div>
            <div style="font-size:12px;color:#777;margin-top:8px;">All Classes:</div>
            <div style="font-size:22px;margin-top:4px;">
                {"  ".join([EMOJIS[e] for e in EMOTIONS])}</div>
            <div style="font-size:10px;color:#aaa;margin-top:2px;">
                {"   ".join(EMOTIONS)}</div>
        </div>""", unsafe_allow_html=True)
    else:
        st.markdown("<div style=\\'text-align:center;color:#bbb;padding:30px 0\\'>"
                    "🔍 Results here</div>", unsafe_allow_html=True)
    st.markdown("</div>", unsafe_allow_html=True)

# ── Success bar ────────────────────────────────────────────────────
if st.session_state.prediction:
    st.markdown("""<div class="success-bar">
        ✅ Emotion detected successfully using FER2013-trained CNN model
    </div>""", unsafe_allow_html=True)

# ── Model status ───────────────────────────────────────────────────
st.markdown("---")
if model is None:
    st.warning("⚠️ No model file found. Running in DEMO mode.")
else:
    st.success("✅ Model loaded successfully.")
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written successfully")

✅ app.py written successfully


In [3]:
import subprocess, threading, time
from pyngrok import ngrok, conf

# ── Install pyngrok if not already installed ──────────────
!pip install pyngrok -q

NGROK_TOKEN = "3Ce0HNz6Ebp9M6lX6gx9JxA7LeB_4JtUw1VHCkuPrHm7NehDC"
ngrok.set_auth_token(NGROK_TOKEN)

# ── Start Streamlit in background ─────────────────────────
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)

# ── Open ngrok tunnel ──────────────────────────────────────
public_url = ngrok.connect(8501)
print(f"\n🚀 App is live at: {public_url}")
print("   Open the link above in your browser!")


🚀 App is live at: NgrokTunnel: "https://extended-primate-fanning.ngrok-free.dev" -> "http://localhost:8501"
   Open the link above in your browser!
